In [5]:
import pandas as pd
import numpy as np
from sklearn.metrics import mean_squared_error, mean_absolute_error
from sklearn.preprocessing import StandardScaler
import xgboost as xgb
from pathlib import Path

def standardize_region_names(df):
    replacements = {
        'Ненецкий авт.округ': 'Ненецкий автономный округ',
        'Hенецкий авт.округ': 'Ненецкий автономный округ',
        '  Ненецкий автономный округ': 'Ненецкий автономный округ',
        'Ямало-Ненецкий авт.округ': 'Ямало-Ненецкий автономный округ',
        'Ямало-Hенецкий авт.округ': 'Ямало-Ненецкий автономный округ',
        '  Ямало-Ненецкий автономный округ': 'Ямало-Ненецкий автономный округ',
        'Ханты-Мансийский авт.округ-Югра': 'Ханты-Мансийский автономный округ - Югра',
        '  Ханты-Мансийский автономный округ - Югра': 'Ханты-Мансийский автономный округ - Югра',
        'Республика Татарстан(Татарстан)': 'Республика Татарстан',
        'Чувашская Республика(Чувашия)': 'Чувашская Республика',
        'Республика Северная Осетия- Алания': 'Республика Северная Осетия-Алания',
        'Oмская область': 'Омская область',
        'Hижегородская область': 'Нижегородская область',
        'г. Севастополь': 'г.Севастополь',
        'г.Москва': 'г.Москва',
        'г.Санкт-Петербург': 'г.Санкт-Петербург',
        'Чукотский авт.округ': 'Чукотский автономный округ',
    }
    df['Регион'] = df['Регион'].replace(replacements).str.strip()
    return df

def clean_numeric_columns(df, target_col):
    df = df.copy()
    numeric_columns = [col for col in df.columns if col not in ['Регион', 'Год', target_col]]
    for col in numeric_columns:
        if df[col].dtype == 'object':
            df[col] = (df[col]
                      .astype(str)
                      .str.replace('\xa0', '')
                      .str.replace(' ', '')
                      .str.replace(',', '.')
                      .str.replace('−', '-')
                      .str.replace('–', '-')
                      )
            df[col] = pd.to_numeric(df[col], errors='coerce')
    df[target_col] = pd.to_numeric(df[target_col], errors='coerce')
    return df

def prepare_features_skr(df):
    df = df.sort_values(['Регион', 'Год'])
    first_year = 2014
    df['lag1_СКР'] = df.groupby('Регион')['СКР'].shift(1)
    df['lag2_СКР'] = df.groupby('Регион')['СКР'].shift(2)
    df['lag1_Браков'] = df.groupby('Регион')['Браков'].shift(1)
    df['lag1_Разводов'] = df.groupby('Регион')['Разводов'].shift(1)
    df['lag1_Число родившихся'] = df.groupby('Регион')['Число родившихся'].shift(1)
    df['СКР_MA2'] = df.groupby('Регион')['СКР'].transform(lambda x: x.rolling(2, min_periods=1).mean())
    df['СКР_MA3'] = df.groupby('Регион')['СКР'].transform(lambda x: x.rolling(3, min_periods=1).mean())
    df['year_trend'] = df['Год'] - first_year
    df['Браков_на_1000'] = df['Браков'] / df['Численность населения'] * 1000
    df['Разводов_на_1000'] = df['Разводов'] / df['Численность населения'] * 1000
    df['Родившихся_на_1000'] = df['Число родившихся'] / df['Численность населения'] * 1000
    df['Преступлений_на_1000'] = df['Кол-во преступлений'] / df['Численность населения'] * 1000
    if 'Введено в действие общей площади жилых домов на 1000 человек населения' in df.columns:
        df['жилье_на_1000'] = df['Введено в действие общей площади жилых домов на 1000 человек населения']
    df['соотношение_браков_разводов'] = df['Браков'] / (df['Разводов'] + 1)
    df['Социально_экономический_индекс'] = (
        df['Средняя ЗП'] / df['Величина прожиточного минимума'] - (df['Уровень бедности'] / 100)
    )
    df['стабильность_семьи'] = df['соотношение_браков_разводов'] / (df['Уровень безработицы'] + 1)
    df['изменение_населения'] = df.groupby('Регион')['Численность населения'].pct_change()
    df['изменение_ВРП'] = df.groupby('Регион')['Валовой региональный продукт на душу населения (ОКВЭД 2)'].pct_change()
    df['изменение_браков'] = df.groupby('Регион')['Браков'].pct_change()
    df['изменение_рождаемости'] = df.groupby('Регион')['Число родившихся'].pct_change()
    numeric_cols = df.select_dtypes(include=[np.number]).columns
    for col in numeric_cols:
        if col not in ['Год']:
            df[col] = df.groupby('Регион')[col].transform(lambda x: x.fillna(x.median()) if not x.isnull().all() else x)
    df = df.fillna(df.median(numeric_only=True))
    return df

df = pd.read_excel("общая_СКР.xlsx")
df = standardize_region_names(df)
df = clean_numeric_columns(df, 'СКР')
df['Год'] = pd.to_numeric(df['Год'], errors='coerce').astype(int)
df = df[(df['Год'] >= 2014) & (df['Год'] <= 2023)]

df_prep = prepare_features_skr(df)

train = df_prep[df_prep['Год'] <= 2022]
test = df_prep[df_prep['Год'] == 2023]

feature_names = [
    'Численность населения', 'Число родившихся', 'Браков', 'Разводов',
    'Введено в действие общей площади жилых домов на 1000 человек населения',
    'жилье_на_1000', 'Кол-во преступлений', 'Уровень безработицы',
    'Уровень бедности', 'Величина прожиточного минимума',
    'Валовой региональный продукт на душу населения (ОКВЭД 2)', 'Средняя ЗП',
    'lag1_СКР', 'lag2_СКР', 'lag1_Браков', 'lag1_Разводов', 'lag1_Число родившихся',
    'СКР_MA2', 'СКР_MA3', 'year_trend', 'Браков_на_1000', 'Разводов_на_1000',
    'Родившихся_на_1000', 'Преступлений_на_1000', 'соотношение_браков_разводов',
    'Социально_экономический_индекс', 'стабильность_семьи',
    'изменение_населения', 'изменение_ВРП', 'изменение_браков', 'изменение_рождаемости'
]
feature_names = [f for f in feature_names if f in df_prep.columns]

X_train = train[feature_names]
y_train = train['СКР']
X_test = test[feature_names]
y_test = test['СКР']

scale_features = [f for f in feature_names if not f.startswith(('lag', 'СКР_MA', 'изменение_', 'год_от_начала'))]
scaler = StandardScaler()
X_train_scaled = X_train.copy()
X_test_scaled = X_test.copy()
X_train_scaled[scale_features] = scaler.fit_transform(X_train[scale_features])
X_test_scaled[scale_features] = scaler.transform(X_test[scale_features])

model = xgb.XGBRegressor(
    max_depth=7,
    learning_rate=0.04,
    n_estimators=350,
    random_state=42,
    n_jobs=-1,
    subsample=0.75,
    colsample_bytree=0.75,
    reg_alpha=0.05,
    reg_lambda=0.8
)
model.fit(X_train_scaled, y_train)

y_pred = model.predict(X_test_scaled)

results = []
for i, (_, row) in enumerate(test.iterrows()):
    actual = y_test.iloc[i]
    pred = y_pred[i]
    abs_err = abs(pred - actual)
    rel_err_pct = (abs_err / actual) * 100
    results.append({
        'Регион': row['Регион'],
        'Прогноз_2023': round(pred, 4),
        'Факт_2023': round(actual, 4),
        'Абсолютная_ошибка': round(abs_err, 4),
        'Относительная_ошибка_%': round(rel_err_pct, 2),
        'RMSE': round(abs_err, 4),
        'MAE': round(abs_err, 4)
    })

df_results = pd.DataFrame(results)
mean_abs = df_results['Абсолютная_ошибка'].mean()
mean_rel = df_results['Относительная_ошибка_%'].mean()
global_rmse = np.sqrt(mean_squared_error(df_results['Факт_2023'], df_results['Прогноз_2023']))
global_mae = mean_absolute_error(df_results['Факт_2023'], df_results['Прогноз_2023'])

summary = pd.DataFrame([{
    'Регион': '=== ИТОГО ===',
    'Прогноз_2023': '',
    'Факт_2023': '',
    'Абсолютная_ошибка': round(mean_abs, 4),
    'Относительная_ошибка_%': round(mean_rel, 2),
    'RMSE': round(global_rmse, 4),
    'MAE': round(global_mae, 4)
}])

final_df = pd.concat([df_results, summary], ignore_index=True)
final_df.to_excel("xgboost_СКР.xlsx", index=False)
print("Сохранено: xgboost_СКР.xlsx")

Сохранено: xgboost_СКР.xlsx
